In [ ]:
# load the geojson data 

Research questions: (1) What are general trends in how wards spend their menu money, and how has this changed over time? (2) Is there a correlation between infrastructure spending and tourism, school performance, demographic hardship, and crime?

Every Chicago ward receives roughly the same annual menu money allocation — about $1.32M per year, or ~$16M over the 2012–2023 period. Because totals are nearly equal by design, the amount of money a ward gets tells us almost nothing. What matters is how aldermen choose to spend it: many small projects spread across the ward, or a handful of large ones concentrated in particular places. This choice correlates strongly with school outcomes, neighborhood stability, and which communities get what kinds of infrastructure.

In [3]:
import json
import numpy as np
import pandas as pd
import altair as alt
from scipy import stats

alt.data_transformers.disable_max_rows()

MENU_CSV   = "data/menu.csv"
SCHOOL_CSV = "data/school.csv"
GEOJSON    = "data/chicago-ward-boundaries.geojson"

RED  = "#c94040"
BLUE = "#1a6faf"
TEAL = "#2a9d8f"
GRAY = "#6b7280"
BG   = "#f8f7f4"

In [4]:
# ── Menu ──────────────────────────────────────────────────────────────────────
menu = pd.read_csv(MENU_CSV)
menu["cost_clean"] = menu["cost"].str.replace("[$,]", "", regex=True).astype(float)

menu_ward = (
    menu.groupby("ward")
    .agg(
        total_spending=("cost_clean", "sum"),
        num_projects=("cost_clean", "count"),
        avg_project=("cost_clean", "mean"),
        median_project=("cost_clean", "median"),
    )
    .reset_index()
)
menu_ward["ward"] = menu_ward["ward"].astype(int)

In [5]:
# ── Category spending share per ward ──────────────────────────────────────────
cat_ward = menu.groupby(["ward", "category"])["cost_clean"].sum().reset_index()
cat_ward["pct"] = (
    cat_ward["cost_clean"]
    / cat_ward.groupby("ward")["cost_clean"].transform("sum")
    * 100
)
cat_ward["ward"] = cat_ward["ward"].astype(int)

In [6]:
# ── Year × ward project counts ────────────────────────────────────────────────
yw = (
    menu.groupby(["year", "ward"])["cost_clean"]
    .agg(["sum", "count"])
    .reset_index()
)
yw.columns = ["year", "ward", "year_spending", "year_projects"]
yw["ward"] = yw["ward"].astype(int)

In [7]:
# ── Ward centroids from GeoJSON ───────────────────────────────────────────────
with open(GEOJSON) as f:
    gj = json.load(f)

ward_info = []
for feat in gj["features"]:
    w = int(feat["properties"]["ward"])
    geom = feat["geometry"]
    rings = (
        geom["coordinates"]
        if geom["type"] == "Polygon"
        else [r for poly in geom["coordinates"] for r in poly]
    )
    pts = [p for ring in rings for p in ring]
    ward_info.append({
        "ward": w,
        "centroid_lon": np.mean([p[0] for p in pts]),
        "centroid_lat": np.mean([p[1] for p in pts]),
    })
ward_df = pd.DataFrame(ward_info)

In [8]:
# ── Schools → assign to nearest ward centroid ─────────────────────────────────
school = pd.read_csv(SCHOOL_CSV)

def assign_ward(lat, lon):
    d = np.sqrt(
        (ward_df["centroid_lat"] - lat) ** 2
        + (ward_df["centroid_lon"] - lon) ** 2
    )
    return int(ward_df.iloc[d.idxmin()]["ward"])

school["ward"] = school.apply(
    lambda r: assign_ward(r["School_Latitude"], r["School_Longitude"]), axis=1
)

school_ward = (
    school.groupby("ward")
    .agg(
        num_schools=("School_ID", "count"),
        avg_attendance=("Student_Attendance_Avg_Pct", "mean"),
        avg_truancy=("Chronic_Truancy_Pct", "mean"),
        avg_mobility=("Mobility_Rate_Pct", "mean"),
        avg_grad_rate=("Graduation_4_Year_School_Pct_Year_2", "mean"),
        avg_suspension=("Suspensions_Per_100_Students_Avg_Pct", "mean"),
    )
    .reset_index()
)

In [9]:
# ── Merge all ward-level tables ───────────────────────────────────────────────
df = menu_ward.merge(school_ward, on="ward").merge(ward_df, on="ward")
df.head()

,ward,total_spending,num_projects,avg_project,median_project,num_schools,avg_attendance,avg_truancy,avg_mobility,avg_grad_rate,avg_suspension,centroid_lon,centroid_lat
0,1,16337504.94,605,27004.140397,6874.11,12,86.9,40.990909,10.540000,77.600000,5.4,-87.685475,41.910678
1,2,16125003.45,596,27055.374916,10971.00,10,86.9,42.240000,7.811111,90.166667,5.4,-87.652733,41.907012
2,3,16238453.39,549,29578.239326,14725.00,16,86.9,62.693750,26.507143,78.840000,5.4,-87.625578,41.824905
3,4,16307731.34,493,33078.562556,22268.38,10,86.9,45.566667,11.600000,84.800000,5.4,-87.606457,41.837073
4,5,15690966.48,287,54672.357073,28888.44,15,86.9,49.380000,11.693333,87.966667,5.4,-87.581698,41.781464


In [10]:
def rp(col_x, col_y):
    v = df[[col_x, col_y]].dropna()
    r, p = stats.pearsonr(v[col_x], v[col_y])
    return r, p

for col, label in [
    ("avg_truancy",  "Truancy"),
    ("avg_mobility", "Mobility"),
    ("avg_grad_rate","Grad Rate"),
]:
    v = df[["num_projects", "avg_project", col]].dropna()
    r_n, p_n = stats.pearsonr(v["num_projects"], v[col])
    r_a, p_a = stats.pearsonr(v["avg_project"],  v[col])
    print(f"{label:12s}  vs # projects: r={r_n:+.3f} p={p_n:.3f}"
          f"  |  vs avg $: r={r_a:+.3f} p={p_a:.3f}")

Truancy       vs # projects: r=-0.177 p=0.218  |  vs avg $: r=+0.162 p=0.262
Mobility      vs # projects: r=-0.339 p=0.016  |  vs avg $: r=+0.309 p=0.029
Grad Rate     vs # projects: r=+0.350 p=0.019  |  vs avg $: r=-0.413 p=0.005


In [11]:
ward_sel = alt.selection_point(fields=["ward"], empty="all")

base_tt = [
    alt.Tooltip("ward:O",         title="Ward"),
    alt.Tooltip("num_projects:Q", title="# Projects",    format=","),
    alt.Tooltip("avg_project:Q",  title="Avg $/Project", format="$,.0f"),
]

In [12]:
r1, p1 = rp("num_projects", "avg_truancy")

s1 = (
    alt.Chart(df)
    .mark_circle(size=90, opacity=0.85, stroke="white", strokeWidth=0.5)
    .encode(
        x=alt.X("num_projects:Q", title="Number of Menu Projects (2012–2023)",
                scale=alt.Scale(zero=False)),
        y=alt.Y("avg_truancy:Q", title="Avg Chronic Truancy Rate (%)",
                scale=alt.Scale(zero=False)),
        color=alt.condition(
            ward_sel,
            alt.Color("avg_truancy:Q", scale=alt.Scale(scheme="orangered"), legend=None),
            alt.value(GRAY + "55"),
        ),
        size=alt.Size("num_schools:Q", scale=alt.Scale(range=[50, 300]),
                      legend=alt.Legend(title="# Schools")),
        tooltip=base_tt + [
            alt.Tooltip("avg_truancy:Q", title="Truancy (%)", format=".1f"),
            alt.Tooltip("num_schools:Q", title="# Schools"),
        ],
    )
    .add_params(ward_sel)
    .properties(
        width=320, height=260,
        title=alt.TitleParams(
            text="More Projects → Lower Truancy",
            subtitle=f"r = {r1:.2f}  p = {p1:.3f}  |  bubble size = # schools",
            fontSize=13, subtitleFontSize=10, subtitleColor=GRAY,
        ),
    )
)

reg1 = s1.transform_regression("num_projects", "avg_truancy").mark_line(
    color=RED, strokeDash=[4, 3], strokeWidth=1.8, opacity=0.7
)

chart1 = s1 + reg1
chart1

alt.LayerChart(...)

In [13]:
r2, p2 = rp("avg_project", "avg_grad_rate")

s2 = (
    alt.Chart(df)
    .mark_circle(size=90, opacity=0.85, stroke="white", strokeWidth=0.5)
    .encode(
        x=alt.X("avg_project:Q", title="Avg Project Size ($)",
                scale=alt.Scale(zero=False), axis=alt.Axis(format="$,.0f")),
        y=alt.Y("avg_grad_rate:Q", title="Avg 4-Yr Grad Rate (%)",
                scale=alt.Scale(zero=False)),
        color=alt.condition(
            ward_sel,
            alt.Color("avg_grad_rate:Q", scale=alt.Scale(scheme="blues", reverse=True),
                      legend=None),
            alt.value(GRAY + "55"),
        ),
        size=alt.Size("num_schools:Q", scale=alt.Scale(range=[50, 300]), legend=None),
        tooltip=base_tt + [
            alt.Tooltip("avg_grad_rate:Q", title="Grad Rate (%)", format=".1f"),
        ],
    )
    .add_params(ward_sel)
    .properties(
        width=320, height=260,
        title=alt.TitleParams(
            text="Larger Projects → Lower Grad Rates",
            subtitle=f"r = {r2:.2f}  p = {p2:.3f}  |  wards with fewer, costlier projects lag",
            fontSize=13, subtitleFontSize=10, subtitleColor=GRAY,
        ),
    )
)

reg2 = s2.transform_regression("avg_project", "avg_grad_rate").mark_line(
    color=BLUE, strokeDash=[4, 3], strokeWidth=1.8, opacity=0.7
)

chart2 = s2 + reg2
chart2

alt.LayerChart(...)

In [14]:
r3, p3 = rp("avg_project", "avg_mobility")

s3 = (
    alt.Chart(df)
    .mark_circle(size=90, opacity=0.85, stroke="white", strokeWidth=0.5)
    .encode(
        x=alt.X("avg_project:Q", title="Avg Project Size ($)",
                scale=alt.Scale(zero=False), axis=alt.Axis(format="$,.0f")),
        y=alt.Y("avg_mobility:Q", title="Avg Student Mobility Rate (%)",
                scale=alt.Scale(zero=False)),
        color=alt.condition(
            ward_sel,
            alt.Color("avg_mobility:Q", scale=alt.Scale(scheme="purples"), legend=None),
            alt.value(GRAY + "55"),
        ),
        size=alt.Size("num_schools:Q", scale=alt.Scale(range=[50, 300]), legend=None),
        tooltip=base_tt + [
            alt.Tooltip("avg_mobility:Q", title="Mobility (%)", format=".1f"),
        ],
    )
    .add_params(ward_sel)
    .properties(
        width=320, height=260,
        title=alt.TitleParams(
            text="Larger Projects → Higher Student Mobility",
            subtitle=f"r = {r3:.2f}  p = {p3:.3f}  |  instability correlates with concentrated spend",
            fontSize=13, subtitleFontSize=10, subtitleColor=GRAY,
        ),
    )
)

reg3 = s3.transform_regression("avg_project", "avg_mobility").mark_line(
    color="#7c3aed", strokeDash=[4, 3], strokeWidth=1.8, opacity=0.7
)

chart3 = s3 + reg3
chart3

alt.LayerChart(...)

In [15]:
bar = (
    alt.Chart(df)
    .mark_bar(cornerRadiusTopLeft=2, cornerRadiusTopRight=2)
    .encode(
        x=alt.X("ward:O", sort=alt.SortField("num_projects", order="descending"),
                title="Ward", axis=alt.Axis(labelFontSize=8)),
        y=alt.Y("num_projects:Q", title="# Projects"),
        color=alt.Color("avg_truancy:Q", scale=alt.Scale(scheme="orangered"),
                        title="Truancy (%)", legend=alt.Legend(orient="right")),
        opacity=alt.condition(ward_sel, alt.value(1.0), alt.value(0.35)),
        tooltip=base_tt + [
            alt.Tooltip("avg_truancy:Q",  title="Truancy (%)",   format=".1f"),
            alt.Tooltip("avg_grad_rate:Q",title="Grad Rate (%)", format=".1f"),
        ],
    )
    .add_params(ward_sel)
    .properties(
        width=980, height=180,
        title=alt.TitleParams(
            text="Project Count per Ward  (color = chronic truancy rate)",
            subtitle="Click a bar to highlight that ward across all charts",
            fontSize=13, subtitleFontSize=10, subtitleColor=GRAY,
        ),
    )
)

bar

alt.Chart(...)

In [16]:
metrics = {
    "Chronic Truancy (%)": "avg_truancy",
    "Student Mobility (%)": "avg_mobility",
    "4-Yr Grad Rate (%)":  "avg_grad_rate",
    "Avg Suspensions":     "avg_suspension",
    "# Schools":           "num_schools",
}

corr_rows = []
for label, col in metrics.items():
    v = df[["num_projects", "avg_project", col]].dropna()
    if len(v) < 10:
        continue
    r_np, p_np = stats.pearsonr(v["num_projects"], v[col])
    r_ap, p_ap = stats.pearsonr(v["avg_project"],  v[col])
    corr_rows.append({"metric": label, "x_var": "# Projects",   "r": r_np, "p": p_np})
    corr_rows.append({"metric": label, "x_var": "Avg Project $", "r": r_ap, "p": p_ap})

corr_df = pd.DataFrame(corr_rows)
corr_df["sig"] = corr_df["p"].apply(lambda p: "p<0.05" if p < 0.05 else "n.s.")

corr_chart = (
    alt.Chart(corr_df)
    .mark_bar(cornerRadiusTopLeft=3, cornerRadiusTopRight=3)
    .encode(
        x=alt.X("r:Q", title="Pearson r", scale=alt.Scale(domain=[-0.55, 0.55])),
        y=alt.Y("metric:N", title=None, sort="-x"),
        color=alt.Color(
            "r:Q",
            scale=alt.Scale(scheme="redblue", domain=[-0.5, 0.5], reverse=True),
            legend=None,
        ),
        row=alt.Row(
            "x_var:N", title=None,
            header=alt.Header(labelFontSize=11, labelFontWeight="bold"),
        ),
        opacity=alt.condition(alt.datum.sig == "p<0.05", alt.value(1.0), alt.value(0.35)),
        tooltip=[
            alt.Tooltip("metric:N", title="Outcome"),
            alt.Tooltip("x_var:N",  title="Menu Variable"),
            alt.Tooltip("r:Q",      title="Pearson r",  format=".3f"),
            alt.Tooltip("p:Q",      title="p-value",    format=".3f"),
            alt.Tooltip("sig:N",    title="Significance"),
        ],
    )
    .properties(
        width=300, height=110,
        title=alt.TitleParams(
            text="Correlation Summary",
            subtitle="Faded bars = not significant (p ≥ 0.05)",
            fontSize=13, subtitleFontSize=10, subtitleColor=GRAY,
        ),
    )
)

corr_chart

/Users/Sarah/opt/anaconda3/lib/python3.8/site-packages/scipy/stats/stats.py:3943: PearsonRNearConstantInputWarning: An input array is nearly constant; the computed correlation coefficent may be inaccurate.
  warnings.warn(PearsonRNearConstantInputWarning())


alt.Chart(...)

In [17]:
geo = (
    alt.Chart(df)
    .mark_circle(stroke="white", strokeWidth=0.8, opacity=0.87)
    .encode(
        longitude="centroid_lon:Q",
        latitude="centroid_lat:Q",
        size=alt.Size("num_projects:Q", scale=alt.Scale(range=[60, 900]),
                      title="# Projects", legend=alt.Legend(orient="bottom-right")),
        color=alt.Color("avg_truancy:Q", scale=alt.Scale(scheme="orangered"),
                        title="Truancy (%)", legend=None),
        tooltip=[
            alt.Tooltip("ward:O",          title="Ward"),
            alt.Tooltip("num_projects:Q",  title="# Projects",    format=","),
            alt.Tooltip("avg_truancy:Q",   title="Truancy (%)",   format=".1f"),
            alt.Tooltip("avg_grad_rate:Q", title="Grad Rate (%)", format=".1f"),
        ],
    )
    .project(type="mercator")
    .properties(
        width=340, height=320,
        title=alt.TitleParams(
            text="Ward Map: Projects & Truancy",
            subtitle="Size = # projects  |  color = truancy rate",
            fontSize=13, subtitleFontSize=10, subtitleColor=GRAY,
        ),
    )
)

geo

alt.Chart(...)

In [18]:
yearly = yw.groupby("year")["year_projects"].sum().reset_index()

trend = (
    alt.Chart(yearly)
    .mark_line(
        color=TEAL, strokeWidth=2.5,
        point=alt.OverlayMarkDef(color=TEAL, size=55),
    )
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("year_projects:Q", title="Total Projects Funded",
                scale=alt.Scale(zero=False)),
        tooltip=[
            alt.Tooltip("year:O",          title="Year"),
            alt.Tooltip("year_projects:Q", title="Total Projects", format=","),
        ],
    )
    .properties(
        width=300, height=180,
        title=alt.TitleParams(
            text="City-Wide Projects per Year",
            subtitle="All 50 wards combined",
            fontSize=13, subtitleFontSize=10, subtitleColor=GRAY,
        ),
    )
)

trend

alt.Chart(...)

In [19]:
ward_order = df.sort_values("num_projects")["ward"].astype(str).tolist()
cw = cat_ward.copy()
cw["ward"] = cw["ward"].astype(str)

heat = (
    alt.Chart(cw)
    .mark_rect()
    .encode(
        x=alt.X("category:N", title=None,
                axis=alt.Axis(labelAngle=-35, labelFontSize=9)),
        y=alt.Y("ward:O", title="Ward  (fewest → most projects)",
                sort=ward_order, axis=alt.Axis(labelFontSize=7)),
        color=alt.Color("pct:Q", title="% of ward spend",
                        scale=alt.Scale(scheme="viridis")),
        tooltip=[
            alt.Tooltip("ward:O",       title="Ward"),
            alt.Tooltip("category:N",   title="Category"),
            alt.Tooltip("pct:Q",        title="Share (%)",  format=".1f"),
            alt.Tooltip("cost_clean:Q", title="$ Spent",    format="$,.0f"),
        ],
    )
    .properties(
        width=440, height=520,
        title=alt.TitleParams(
            text="Spending Category Mix by Ward",
            subtitle="Sorted by # projects — reveals how wards prioritize spend",
            fontSize=13, subtitleFontSize=10, subtitleColor=GRAY,
        ),
    )
)

heat

alt.Chart(...)